[← Back to 05 - Subqueries to CTEs](<../05-subqueries-to-ctes.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

# 05 - Subqueries to CTEs: Solutions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/solutions/05-subqueries-to-ctes-solutions.ipynb)

Try each exercise in the lesson notebook first. If your query differs from the one here but returns the same rows, it may well be just as correct, and seeing why is a useful exercise in itself.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80),
    (104, 2, 90),
    (105, 3, 140);

4 rows inserted
5 rows inserted


In [3]:
%%sql
CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    employee_name TEXT NOT NULL,
    manager_id INTEGER
);
INSERT INTO employees VALUES
    (1, 'Ngozi', NULL),
    (2, 'Femi', 1),
    (3, 'Tunde', 1),
    (4, 'Zainab', 2);

4 rows inserted


## 1. Find the top spender

In [4]:
%%sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.credit_limit,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name,
        c.credit_limit
)
SELECT
    customer_id,
    customer_name,
    total_spent
FROM customer_totals
ORDER BY
    total_spent DESC
LIMIT 1;

customer_id,customer_name,total_spent
2,Bola,170


Bola, whose two orders add up to 170, more than anyone else's, even though Ada was the one flagged in the original credit-limit bug. One caution with `LIMIT 1`: if two customers tied for the top total, it would return one of them and silently drop the other. To keep ties, compare against the maximum instead: `WHERE total_spent = (SELECT MAX(total_spent) FROM customer_totals)`.

## 2. Customers with no orders, without a join

In [5]:
%%sql
SELECT
    customer_id,
    customer_name
FROM customers
WHERE customer_id NOT IN (SELECT customer_id FROM orders)
ORDER BY
    customer_id;

customer_id,customer_name
4,Dami


Dami. `NOT IN` is safe here because `orders.customer_id` is declared `NOT NULL`; if the subquery could return a `NULL`, `NOT IN` would return no rows at all, the trap `02` warned about. `NOT EXISTS` doesn't have that problem, so many people use it by default:

In [6]:
%%sql
SELECT
    c.customer_id,
    c.customer_name
FROM customers AS c
WHERE NOT EXISTS (
    SELECT 1
    FROM orders AS o
    WHERE o.customer_id = c.customer_id
)
ORDER BY
    c.customer_id;

customer_id,customer_name
4,Dami


`NOT EXISTS` is a correlated subquery: for each customer, it asks whether any order points at them, and keeps the customer when none does.

## 3. Count each manager's direct reports

In [7]:
%%sql
SELECT
    e.employee_name,
    (
        SELECT COUNT(*)
        FROM employees AS r
        WHERE r.manager_id = e.employee_id
    ) AS direct_reports
FROM employees AS e
ORDER BY
    e.employee_id;

employee_name,direct_reports
Ngozi,2
Femi,1
Tunde,0
Zainab,0


Ngozi has two direct reports, Femi has one, and Tunde and Zainab have none. Because the subquery runs once per employee, everyone appears, including the zeros, which a `GROUP BY manager_id` on its own would have missed (nobody's `manager_id` is Tunde's, so no group for Tunde would exist).

## 4. Stretch: customers above the average customer (AdventureWorks)

This answer runs on SQL Server, so set up AdventureWorks first. In Colab that takes a few minutes the first time, and on your own machine it only checks the connection.

In [8]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


In [9]:
%%tsql
WITH customer_totals AS (
    SELECT
        CustomerID,
        SUM(TotalDue) AS total_spent
    FROM Sales.SalesOrderHeader
    GROUP BY
        CustomerID
),
average_total AS (
    SELECT AVG(total_spent) AS average_spent
    FROM customer_totals
)
SELECT
    COUNT(*) AS customers_with_orders,
    SUM(CASE WHEN ct.total_spent > a.average_spent THEN 1 ELSE 0 END) AS above_average
FROM customer_totals AS ct
CROSS JOIN average_total AS a;

customers_with_orders,above_average
19119,1927


1,927 of the 19,119 customers who've ordered spent more than the average customer, about one in ten, the same skew you saw with individual orders: a small group of large customers pulls the average far above what a typical customer spends. That's why "above average" is often a less useful line than it sounds, and `09 - Thinking in Metrics` comes back to what an "average" should even include.

The `CASE WHEN ... THEN 1 ELSE 0 END` inside `SUM` counts the rows that meet a condition. Putting the average in its own CTE matters on SQL Server: writing `SUM(CASE WHEN total_spent > (SELECT AVG(total_spent) FROM customer_totals) ...)` directly fails, because SQL Server won't put a subquery inside an aggregate.

---

[← Back to 05 - Subqueries to CTEs](<../05-subqueries-to-ctes.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](../LICENSE).